In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import statsmodels.api as sm

In [50]:
import pandas as pd

amz = pd.read_csv(r"C:\Users\thaon\Downloads\amz_report_mysql_ready.csv")
amz.head()

,order_id,order_date,status,fulfilment,sales_channel,ship_service_level,style,sku,category,size,...,qty,currency,amount,ship_city,ship_state,ship_postal_code,ship_country,promotion_ids,b2b,fulfilled_by
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,...,0,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,0,Easy Ship
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,...,1,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,0,Easy Ship
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,...,1,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,1,NaN
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,...,0,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,0,Easy Ship
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,...,1,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,0,NaN


In [51]:
# Check Data Types and Missing Values
print(amz.dtypes)

order_id               object
order_date             object
status                 object
fulfilment             object
sales_channel          object
ship_service_level     object
style                  object
sku                    object
category               object
size                   object
asin                   object
courier_status         object
qty                     int64
currency               object
amount                float64
ship_city              object
ship_state             object
ship_postal_code      float64
ship_country           object
promotion_ids          object
b2b                     int64
fulfilled_by           object
dtype: object


In [52]:
cat_cols = [
    "status",
    "fulfilment",
    "fulfilled_by",
    "sales_channel",
    "ship_service_level",
    "style",
    "category",
    "size",
    "courier_status",
    "currency",
    "ship_state",
    "ship_country",
    "b2b"
]
for col in cat_cols:
    amz[col] = amz[col].astype("category")

amz["order_date"] = pd.to_datetime(amz["order_date"])

amz["ship_postal_code"] = (
    amz["ship_postal_code"]
    .fillna("")
    .astype(str)
    .str.replace(".0", "", regex=False)
)

amz["b2b"] = amz["b2b"].astype(bool)

In [53]:
print(amz.dtypes)

order_id                      object
order_date            datetime64[ns]
status                      category
fulfilment                  category
sales_channel               category
ship_service_level          category
style                       category
sku                           object
category                    category
size                        category
asin                          object
courier_status              category
qty                            int64
currency                    category
amount                       float64
ship_city                     object
ship_state                  category
ship_postal_code              object
ship_country                category
promotion_ids                 object
b2b                             bool
fulfilled_by                category
dtype: object


In [54]:
amz.describe().T

,count,mean,min,25%,50%,75%,max,std
order_date,128975,2022-05-12 11:49:27.951928576,2022-03-31 00:00:00,2022-04-20 00:00:00,2022-05-10 00:00:00,2022-06-04 00:00:00,2022-06-29 00:00:00,NaN
qty,128975.0,0.904431,0.0,1.0,1.0,1.0,15.0,0.313354
amount,121180.0,648.561465,0.0,449.0,605.0,788.0,5584.0,281.211687


In [55]:
# descriptive 
amz.describe(include = 'O')

,order_id,sku,asin,ship_city,ship_postal_code,promotion_ids
count,128975,128975,128975,128942,128975,79822
unique,120378,7195,7190,8955,9460,5787
top,171-5057375-2831560,JNE3797-KR-L,B09SDXFFQ1,BENGALURU,201301,IN Core Free Shipping 2015/04/08 23-48-5-108
freq,12,773,773,11217,1006,46100


In [56]:
# 1. Check duplicates
amz.duplicated().sum()

6

In [57]:
duplicates = amz[amz.duplicated(keep=False)]
duplicates.sort_values(by="order_id")

,order_id,order_date,status,fulfilment,sales_channel,ship_service_level,style,sku,category,size,...,qty,currency,amount,ship_city,ship_state,ship_postal_code,ship_country,promotion_ids,b2b,fulfilled_by
85790,171-3249942-2207542,2022-05-03,Shipped,Amazon,Amazon.in,Expedited,SET323,SET323-KR-NP-XL,Set,XL,...,1,INR,939.0,PUNE,MAHARASHTRA,411057,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN
85791,171-3249942-2207542,2022-05-03,Shipped,Amazon,Amazon.in,Expedited,SET323,SET323-KR-NP-XL,Set,XL,...,1,INR,939.0,PUNE,MAHARASHTRA,411057,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN
79844,171-9628368-5329958,2022-05-07,Cancelled,Amazon,Amazon.in,Expedited,J0329,J0329-KR-L,kurta,L,...,0,NaN,NaN,ERNAKULAM,KERALA,682017,IN,NaN,False,NaN
79845,171-9628368-5329958,2022-05-07,Cancelled,Amazon,Amazon.in,Expedited,J0329,J0329-KR-L,kurta,L,...,0,NaN,NaN,ERNAKULAM,KERALA,682017,IN,NaN,False,NaN
86418,405-8669298-3850736,2022-05-03,Shipped,Amazon,Amazon.in,Expedited,MEN5025,MEN5025-KR-XXXL,kurta,3XL,...,1,INR,533.0,GHAZIABAD,UTTAR PRADESH,201010,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN
86419,405-8669298-3850736,2022-05-03,Shipped,Amazon,Amazon.in,Expedited,MEN5025,MEN5025-KR-XXXL,kurta,3XL,...,1,INR,533.0,GHAZIABAD,UTTAR PRADESH,201010,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN
30660,406-0372545-6086735,2022-04-12,Cancelled,Amazon,Amazon.in,Expedited,SET197,SET197-KR-NP-L,Set,L,...,0,NaN,NaN,Siliguri,WEST BENGAL,734008,IN,NaN,False,NaN
30661,406-0372545-6086735,2022-04-12,Cancelled,Amazon,Amazon.in,Expedited,SET197,SET197-KR-NP-L,Set,L,...,0,NaN,NaN,Siliguri,WEST BENGAL,734008,IN,NaN,False,NaN
98954,407-4853873-4978725,2022-06-22,Shipped,Amazon,Amazon.in,Expedited,J0230,J0230-SKD-M,Set,M,...,1,INR,1163.0,Zirakpur,Punjab,140603,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN
98955,407-4853873-4978725,2022-06-22,Shipped,Amazon,Amazon.in,Expedited,J0230,J0230-SKD-M,Set,M,...,1,INR,1163.0,Zirakpur,Punjab,140603,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN


In [58]:
amz = amz.drop_duplicates()

In [59]:
amz.duplicated().sum()

0

In [60]:
#droping columns
amz.drop(columns= ['fulfilled_by', 'ship_country', 'currency', 'sales_channel'], inplace = True)

In [61]:
# 2. Check missing values
amz.isnull().sum()

order_id                  0
order_date                0
status                    0
fulfilment                0
ship_service_level        0
style                     0
sku                       0
category                  0
size                      0
asin                      0
courier_status         6872
qty                       0
amount                 7792
ship_city                33
ship_state               33
ship_postal_code          0
promotion_ids         49150
b2b                       0
dtype: int64

In [62]:
num_cols = ["qty", "amount"]
display(amz[num_cols].describe())
cat_cols = [
    "courier_status",
    "ship_city",
    "ship_state",
    "promotion_ids"
]
display(amz[cat_cols].describe())

,qty,amount
count,128969.00000,121177.000000
mean,0.90445,648.555776
std,0.31333,281.209851
min,0.00000,0.000000
25%,1.00000,449.000000
50%,1.00000,605.000000
75%,1.00000,788.000000
max,15.00000,5584.000000


,courier_status,ship_city,ship_state,promotion_ids
count,122097,128936,128936,79819
unique,3,8955,69,5787
top,Shipped,BENGALURU,MAHARASHTRA,IN Core Free Shipping 2015/04/08 23-48-5-108
freq,109484,11216,22259,46097


QTY: most custumers purchase a single product per order. These presence of orders with equal to zero suggests cancelled or unfulfilled transactions that should be investigated
Amount: about 7792 missing values, mean is higher than median ( 648.56 > 605) so the data is right-skewed meaing a relatively small number of high value purchases increase the average order value: so we cannot replace missing value automatically because they likely correspond to cancelled or imcomplete orders
Courier Status: the vast majority of orders are shipped, indicating that most of transactions were successfully processed through logistics network
Ship City: The company serves customer across 8955 cities. This is a board geographic coverage, and Bengaluru is the biggest customer market.
Ship State: Orders are distributed across 69 states, with Maharashtra contributing the largest share of sales, suggesting it is one of the company's strongest regional markets.
Promotion ids: Not every order uses a promotion, as appromximately 38% missing values. Among promotional orders, Free Shipping is by far the most frequently applied promotion, suggesting shipping incentives play an important role in customer purchases.

In [63]:
amz[amz["amount"].isna()]["status"].value_counts()

status
Cancelled                        7563
Shipped                           208
Shipped - Delivered to Buyer        8
Shipping                            8
Shipped - Returned to Seller        3
Pending                             2
Pending - Waiting for Pick Up       0
Shipped - Damaged                   0
Shipped - Lost in Transit           0
Shipped - Out for Delivery          0
Shipped - Picked Up                 0
Shipped - Rejected by Buyer         0
Shipped - Returning to Seller       0
Name: count, dtype: int64

Total missing values are 7792 with 97.1% missing amount are associated with cancelled orders. This strongly suggests that the missing values are business-relate, not data-quality errors. Therefore, these values were retained as missing instead of being imputed. 

In [64]:
pd.crosstab(
    amz["status"],
    amz["courier_status"],
    dropna=False
)

courier_status,Cancelled,Shipped,Unshipped
status,,,
Cancelled,5837,0,5631
Pending,2,10,646
Pending - Waiting for Pick Up,0,0,281
Shipped,93,77593,115
Shipped - Damaged,0,1,0
Shipped - Delivered to Buyer,0,28761,0
Shipped - Lost in Transit,0,5,0
Shipped - Out for Delivery,0,35,0
Shipped - Picked Up,0,973,0


the missing courier_status values are almost entirely associated with cancelled orders. Therefore, orders were cancelled before courier information was recorded, or the courier assigment process never started. Therefore, the missing values are expected business outcomes, not data quality problems.

97.1% belong to Cancelled orders.
The remaining few belong to orders that were never completed or are in transitional statuses.

For sales analysis purpose, we will create sales table with clean values.

In [65]:
sales = amz[amz["amount"].notna()]

In [66]:
sales.isnull().sum()

order_id                  0
order_date                0
status                    0
fulfilment                0
ship_service_level        0
style                     0
sku                       0
category                  0
size                      0
asin                      0
courier_status         5136
qty                       0
amount                    0
ship_city                31
ship_state               31
ship_postal_code          0
promotion_ids         41698
b2b                       0
dtype: int64

In [67]:
sales[sales["courier_status"].isna()]["status"].value_counts()

status
Cancelled                        5136
Pending                             0
Pending - Waiting for Pick Up       0
Shipped                             0
Shipped - Damaged                   0
Shipped - Delivered to Buyer        0
Shipped - Lost in Transit           0
Shipped - Out for Delivery          0
Shipped - Picked Up                 0
Shipped - Rejected by Buyer         0
Shipped - Returned to Seller        0
Shipped - Returning to Seller       0
Shipping                            0
Name: count, dtype: int64

In [68]:
sales["courier_status"] = (
    sales["courier_status"]
    .cat.add_categories(["Not Assigned"])
    .fillna("Not Assigned")
)

C:\Users\thaon\AppData\Local\Temp\ipykernel_31592\3581039516.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales["courier_status"] = (


In [69]:
sales.isnull().sum()

order_id                  0
order_date                0
status                    0
fulfilment                0
ship_service_level        0
style                     0
sku                       0
category                  0
size                      0
asin                      0
courier_status            0
qty                       0
amount                    0
ship_city                31
ship_state               31
ship_postal_code          0
promotion_ids         41698
b2b                       0
dtype: int64

In [70]:
sales["promotion_used"] = sales["promotion_ids"].notna().map({
    True: "Yes",
    False: "No"
})

C:\Users\thaon\AppData\Local\Temp\ipykernel_31592\814520710.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales["promotion_used"] = sales["promotion_ids"].notna().map({


In [73]:
sales["ship_city"] = sales["ship_city"].fillna("Unknown") # object col
sales["ship_state"] = (
    sales["ship_state"]
    .cat.add_categories("Unknown")
    .fillna("Unknown")
) # category col

C:\Users\thaon\AppData\Local\Temp\ipykernel_31592\2204045066.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales["ship_city"] = sales["ship_city"].fillna("Unknown") # object col
C:\Users\thaon\AppData\Local\Temp\ipykernel_31592\2204045066.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales["ship_state"] = (


In [74]:
sales.isnull().sum()

order_id                  0
order_date                0
status                    0
fulfilment                0
ship_service_level        0
style                     0
sku                       0
category                  0
size                      0
asin                      0
courier_status            0
qty                       0
amount                    0
ship_city                 0
ship_state                0
ship_postal_code          0
promotion_ids         41698
b2b                       0
promotion_used            0
dtype: int64

In [75]:
sales.columns

Index(['order_id', 'order_date', 'status', 'fulfilment', 'ship_service_level',
       'style', 'sku', 'category', 'size', 'asin', 'courier_status', 'qty',
       'amount', 'ship_city', 'ship_state', 'ship_postal_code',
       'promotion_ids', 'b2b', 'promotion_used'],
      dtype='object')

In [76]:
sales.to_csv("amazon_sales_cleaned.csv", index=False)

In [77]:
import os

os.getcwd()

'C:\\Users\\thaon\\Downloads\\Amazon Sale Report.csv'